# 03 — Competition-faithful baselines and official scoring

Baselines:
- Persistence / last-observation logistic regression
- GRU
- GRU-D

Protocol:
1. Use only Set A outcomes for development.
2. Select neural training duration and Event-1 threshold using an internal Set-A validation split.
3. Refit preprocessing on all A and retrain final models on all A with frozen settings.
4. Generate B/C predictions without their outcomes.
5. **Only after predictions are frozen**, load the now-public B/C outcomes and score them.
6. Report C as the historical final-test comparison.

This reproduces the original information boundary as closely as possible today. The original competition additionally allowed a limited number of B submissions whose returned scores could guide later entries; we deliberately do not exploit `Outcomes-b.txt` for tuning.


In [1]:
import sys, copy, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Start Jupyter from repository root or notebooks/.")
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.physionet import load_split
from src.data.preprocessing import Physionet2012Dataset, build_vocab

RAW_DIR = PROJECT_ROOT / "data" / "raw"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
plt.rcParams["figure.dpi"] = 110
print("Project root:", PROJECT_ROOT)

import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)


Project root: C:\Users\guido\git_projects\icu-continuous-forecasting
Device: cpu


In [2]:
A=load_split(RAW_DIR,split="set-a")
B=load_split(RAW_DIR,split="set-b")
yA=np.asarray([int(r.label) for r in A])

idx=np.arange(len(A))
tr_idx,va_idx=train_test_split(idx,test_size=.20,random_state=SEED,stratify=yA)
A_tr=[A[i] for i in tr_idx]; A_va=[A[i] for i in va_idx]

dev_vocab=build_vocab(A_tr)
dev_tr_ds=Physionet2012Dataset(A_tr,dev_vocab); dev_tr_ds.fit_normalizer()
dev_va_ds=copy.deepcopy(dev_tr_ds); dev_va_ds.records=A_va

def mat(ds):
    b=ds.collate(list(range(len(ds.records))))
    return {"values":np.nan_to_num(b.values.numpy().astype("float32"),nan=0.0),
            "mask":b.mask.numpy().astype("float32"),
            "delta_t":b.delta_t.numpy().astype("float32"),
            "static":b.static.numpy().astype("float32"),
            "y":b.labels.numpy().astype("float32")}
dev_tr,dev_va=mat(dev_tr_ds),mat(dev_va_ds)


## 1. Official Challenge metrics

Event 1 uses the submitted **binary prediction**, not a threshold optimized on B/C. We select our threshold on Set-A internal validation and freeze it.

Event 2 is `H/D` over ten equal-sized risk groups. Lower is better.


In [3]:
def event1(y,pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    tp=np.sum((pred==1)&(y==1)); fp=np.sum((pred==1)&(y==0)); fn=np.sum((pred==0)&(y==1))
    se=tp/(tp+fn) if tp+fn else 0.
    ppv=tp/(tp+fp) if tp+fp else 0.
    return min(se,ppv),se,ppv

def choose_threshold(y,risk):
    cand=np.unique(np.r_[0.,risk,1.])
    scored=[(event1(y,risk>=t)[0],-abs(t-.5),float(t)) for t in cand]
    return max(scored)[2]

def event2(y,risk):
    y=np.asarray(y,float); risk=np.asarray(risk,float)
    order=np.argsort(risk,kind="mergesort")
    groups=np.array_split(order,10)
    H=0.; pis=[]
    for ids in groups:
        N=len(ids); O=y[ids].sum(); pi=risk[ids].mean(); E=N*pi
        H+=(O-E)**2/(N*pi*(1-pi)+.001); pis.append(pi)
    D=pis[-1]-pis[0]
    return np.inf if D<=0 else H/D

def metrics(y,risk,thr):
    pred=(risk>=thr).astype(int); s,se,ppv=event1(y,pred)
    return {"AUROC":roc_auc_score(y,risk),"AUPRC":average_precision_score(y,risk),
            "Event1":s,"Sensitivity":se,"PPV":ppv,"Threshold":thr,"Event2":event2(y,risk)}


## 2. Models and training utilities


In [4]:
BATCH=128; MAX_EPOCHS=30; PATIENCE=6; LR=1e-3

def batches(data,bs=BATCH,shuffle=False):
    ids=np.arange(len(data["y"]))
    if shuffle: np.random.shuffle(ids)
    for s in range(0,len(ids),bs):
        ii=ids[s:s+bs]
        yield {k:torch.as_tensor(v[ii],device=DEVICE) for k,v in data.items()}

@torch.no_grad()
def predict(model,data):
    model.eval(); out=[]
    for b in batches(data,shuffle=False):
        out.append(torch.sigmoid(model(b)).cpu().numpy())
    return np.concatenate(out)

def develop(model):
    model=model.to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=LR); lossfn=nn.BCEWithLogitsLoss()
    best=None; best_ap=-1; best_epoch=1; bad=0; hist=[]
    for ep in range(1,MAX_EPOCHS+1):
        model.train(); ls=[]
        for b in batches(dev_tr,shuffle=True):
            opt.zero_grad(); z=model(b); loss=lossfn(z,b["y"]); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.); opt.step(); ls.append(loss.item())
        r=predict(model,dev_va); ap=average_precision_score(dev_va["y"],r); auc=roc_auc_score(dev_va["y"],r)
        hist.append((ep,np.mean(ls),auc,ap))
        print(f"{ep:02d} loss={np.mean(ls):.4f} val_AUROC={auc:.4f} val_AUPRC={ap:.4f}")
        if ap>best_ap+1e-5:
            best_ap=ap; best_epoch=ep; best=copy.deepcopy(model.state_dict()); bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best)
    return model,best_epoch,pd.DataFrame(hist,columns=["epoch","loss","AUROC","AUPRC"])

def train_fixed(model,data,epochs):
    model=model.to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=LR); lossfn=nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        model.train()
        for b in batches(data,shuffle=True):
            opt.zero_grad(); loss=lossfn(model(b),b["y"]); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.); opt.step()
    return model

class GRU(nn.Module):
    def __init__(self,D,S,H=64):
        super().__init__(); self.gru=nn.GRU(2*D,H,batch_first=True)
        self.head=nn.Sequential(nn.Linear(H+S,64),nn.ReLU(),nn.Dropout(.2),nn.Linear(64,1))
    def forward(self,b):
        _,h=self.gru(torch.cat([b["values"],b["mask"]],-1))
        return self.head(torch.cat([h[-1],b["static"]],-1)).squeeze(-1)

class GRUD(nn.Module):
    def __init__(self,D,S,H=64):
        super().__init__(); self.H=H; self.gx=nn.Linear(D,D); self.gh=nn.Linear(D,H)
        self.cell=nn.GRUCell(2*D,H)
        self.head=nn.Sequential(nn.Linear(H+S,64),nn.ReLU(),nn.Dropout(.2),nn.Linear(64,1))
    def forward(self,b):
        x,m,dt=b["values"],b["mask"],b["delta_t"]; B,T,D=x.shape
        h=torch.zeros(B,self.H,device=x.device); last=torch.zeros(B,D,device=x.device)
        for t in range(T):
            gx=torch.exp(-torch.relu(self.gx(dt[:,t]))); gh=torch.exp(-torch.relu(self.gh(dt[:,t])))
            h=gh*h; xd=gx*last
            xt=m[:,t]*x[:,t]+(1-m[:,t])*xd
            last=m[:,t]*x[:,t]+(1-m[:,t])*last
            h=self.cell(torch.cat([xt,m[:,t]],-1),h)
        return self.head(torch.cat([h,b["static"]],-1)).squeeze(-1)


## 3. Develop GRU and GRU-D on Set A only

The chosen epoch count and Event-1 threshold are frozen before B/C scoring.


In [5]:
D=dev_tr["values"].shape[-1]; S=dev_tr["static"].shape[-1]

torch.manual_seed(SEED)
gru_dev,gru_epochs,gru_hist=develop(GRU(D,S))
gru_val=predict(gru_dev,dev_va); gru_thr=choose_threshold(dev_va["y"],gru_val)

torch.manual_seed(SEED)
grud_dev,grud_epochs,grud_hist=develop(GRUD(D,S))
grud_val=predict(grud_dev,dev_va); grud_thr=choose_threshold(dev_va["y"],grud_val)

display(pd.DataFrame([
    {"model":"GRU","epochs":gru_epochs,"Event1_threshold":gru_thr},
    {"model":"GRU-D","epochs":grud_epochs,"Event1_threshold":grud_thr},
]).set_index("model").round(4))


01 loss=0.5691 val_AUROC=0.6189 val_AUPRC=0.1978
02 loss=0.3952 val_AUROC=0.7386 val_AUPRC=0.3775
03 loss=0.3610 val_AUROC=0.7675 val_AUPRC=0.4083
04 loss=0.3428 val_AUROC=0.8090 val_AUPRC=0.4391
05 loss=0.3311 val_AUROC=0.8339 val_AUPRC=0.4618
06 loss=0.3196 val_AUROC=0.8435 val_AUPRC=0.4879
07 loss=0.3104 val_AUROC=0.8508 val_AUPRC=0.5097
08 loss=0.3045 val_AUROC=0.8459 val_AUPRC=0.5091
09 loss=0.3034 val_AUROC=0.8561 val_AUPRC=0.5275
10 loss=0.2973 val_AUROC=0.8545 val_AUPRC=0.5309
11 loss=0.2893 val_AUROC=0.8603 val_AUPRC=0.5562
12 loss=0.2827 val_AUROC=0.8426 val_AUPRC=0.5238
13 loss=0.2839 val_AUROC=0.8615 val_AUPRC=0.5618
14 loss=0.2778 val_AUROC=0.8599 val_AUPRC=0.5502
15 loss=0.2722 val_AUROC=0.8599 val_AUPRC=0.5604
16 loss=0.2666 val_AUROC=0.8580 val_AUPRC=0.5602
17 loss=0.2553 val_AUROC=0.8631 val_AUPRC=0.5613
18 loss=0.2509 val_AUROC=0.8591 val_AUPRC=0.5569
19 loss=0.2453 val_AUROC=0.8622 val_AUPRC=0.5570
01 loss=0.5522 val_AUROC=0.7088 val_AUPRC=0.3341
02 loss=0.3554 val_A

,epochs,Event1_threshold
model,,
GRU,13,0.3791
GRU-D,13,0.3127


## 4. Refit preprocessing and final models on all labelled Set A


In [6]:
fv=build_vocab(A)
A_ds=Physionet2012Dataset(A,fv); A_ds.fit_normalizer()
B_ds=Physionet2012Dataset(B,fv,normalizer=A_ds.normalizer)

A_all=mat(A_ds)
B_x=mat(B_ds)

D=A_all["values"].shape[-1]; S=A_all["static"].shape[-1]

torch.manual_seed(SEED)
gru=train_fixed(GRU(D,S),A_all,gru_epochs)
torch.manual_seed(SEED)
grud=train_fixed(GRUD(D,S),A_all,grud_epochs)


## 5. Persistence baseline

Threshold selection is also restricted to A-development-validation. The final logistic model is then fit on all A.


In [7]:
def last_features(d):
    x,m=d["values"],d["mask"]; B,T,D=x.shape
    last=np.zeros((B,D),np.float32); ever=(m.sum(1)>0).astype(np.float32)
    for i in range(B):
        for j in range(D):
            ids=np.flatnonzero(m[i,:,j]>0)
            if len(ids): last[i,j]=x[i,ids[-1],j]
    return np.c_[last,ever,d["static"]]

p_dev=LogisticRegression(max_iter=2000,class_weight="balanced",random_state=SEED)
p_dev.fit(last_features(dev_tr),dev_tr["y"])
p_val=p_dev.predict_proba(last_features(dev_va))[:,1]
p_thr=choose_threshold(dev_va["y"],p_val)

p_final=LogisticRegression(max_iter=2000,class_weight="balanced",random_state=SEED)
p_final.fit(last_features(A_all),A_all["y"])


,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

## 6. Freeze predictions BEFORE reading B/C outcomes

This cell creates the exact submission-like outputs: `RecordID, Prediction, Risk`.

Run it before the scoring cell below.


In [8]:
# Set C was hidden during the Challenge. Load it only now, after every
# development choice and final model fit has been frozen.
C=load_split(RAW_DIR,split="set-c")
C_ds=Physionet2012Dataset(C,fv,normalizer=A_ds.normalizer)
C_x=mat(C_ds)

risk_B={
 "Persistence":p_final.predict_proba(last_features(B_x))[:,1],
 "GRU":predict(gru,B_x),
 "GRU-D":predict(grud,B_x)}
risk_C={
 "Persistence":p_final.predict_proba(last_features(C_x))[:,1],
 "GRU":predict(gru,C_x),
 "GRU-D":predict(grud,C_x)}
thr={"Persistence":p_thr,"GRU":gru_thr,"GRU-D":grud_thr}

def submission(records,risk,t):
    return pd.DataFrame({"RecordID":[r.record_id for r in records],
                         "Prediction":(risk>=t).astype(int),"Risk":risk})

submissions_B={m:submission(B,r,thr[m]) for m,r in risk_B.items()}
submissions_C={m:submission(C,r,thr[m]) for m,r in risk_C.items()}

for m in submissions_C:
    assert submissions_C[m]["Risk"].between(0,1).all()
print("Predictions frozen. B/C outcomes have not been read by this notebook up to this point.")
display(submissions_C["GRU"].head())


Predictions frozen. B/C outcomes have not been read by this notebook up to this point.


,RecordID,Prediction,Risk
0,152871,0,0.121379
1,152873,0,0.076577
2,152875,0,0.143918
3,152878,0,0.059019
4,152882,0,0.010247


## 7. Retrospective official scoring

**Information firewall ends here.** The next cell reads the now-public `Outcomes-b.txt` and
`Outcomes-c.txt` solely to reproduce scoring that PhysioNet performed server-side in 2012.

Nothing after this point may be used to retune the models if we want to preserve the final C result as a clean historical comparison.


In [10]:
def read_outcomes(path):
    df = pd.read_csv(path, dtype={"RecordID": str})
    df["RecordID"] = df["RecordID"].str.strip()

    assert df["RecordID"].is_unique, f"Duplicate RecordID in {path.name}"

    return df.set_index("RecordID")["In-hospital_death"]


yB_map = read_outcomes(RAW_DIR / "Outcomes-b.txt")
yC_map = read_outcomes(RAW_DIR / "Outcomes-c.txt")


def aligned_y(records, outcome_map):
    record_ids = [str(r.record_id).strip() for r in records]

    missing = sorted(set(record_ids) - set(outcome_map.index))
    extra = sorted(set(outcome_map.index) - set(record_ids))

    assert not missing, f"Missing outcome IDs: {missing[:10]}"
    assert not extra, f"Outcome IDs without records: {extra[:10]}"

    return np.asarray(
        [int(outcome_map.loc[rid]) for rid in record_ids],
        dtype=np.int64,
    )


yB = aligned_y(B, yB_map)
yC = aligned_y(C, yC_map)

print("B:", len(yB), "deaths:", yB.sum())
print("C:", len(yC), "deaths:", yC.sum())


rows = []

for m in ["Persistence", "GRU", "GRU-D"]:
    mb = metrics(yB, risk_B[m], thr[m])
    mc = metrics(yC, risk_C[m], thr[m])

    rows.append({"Model": m, "Set": "B", **mb})
    rows.append({"Model": m, "Set": "C", **mc})

results = pd.DataFrame(rows)
display(results.round(4))

B: 4000 deaths: 568
C: 4000 deaths: 585


,Model,Set,AUROC,AUPRC,Event1,Sensitivity,PPV,Threshold,Event2
0,Persistence,B,0.8515,0.5155,0.4966,0.5088,0.4966,0.7609,1978.1065
1,Persistence,C,0.8518,0.5054,0.4769,0.4769,0.4947,0.7609,1964.5257
2,GRU,B,0.8512,0.5147,0.5180,0.5317,0.5180,0.3791,53.9951
3,GRU,C,0.8457,0.5165,0.4974,0.4974,0.5052,0.3791,37.0932
4,GRU-D,B,0.8374,0.4875,0.4613,0.5141,0.4613,0.3127,472.6651
5,GRU-D,C,0.8391,0.5048,0.4817,0.5385,0.4817,0.3127,377.8176


## 8. Historical comparison

For the **official Challenge comparison**, use the **Set C Event 1 / Event 2** rows above.

Published final Event-1 reference points include the winning Johnson et al. score of **0.5353**.
The project should present historical scores as context, while making clear that our modern Python
training implementation is a retrospective reproduction rather than an original 2012 submission.

AUROC and AUPRC are additional modern metrics; they were not the official ranking metrics.


## 9. Critical reproducibility note

The original Challenge allowed participants a limited number of submissions to **Set B**, and PhysioNet returned B scores. Those scores could influence later algorithm revisions. We cannot reconstruct every team's submission history.

Our protocol is therefore deliberately stricter:

```text
A outcomes → development + final training
B outcomes → retrospective score only
C outcomes → retrospective FINAL score only
```

This exactly preserves the crucial label-information boundary and makes the Set-C comparison clean and reproducible. It does **not** claim to recreate the interactive sequence of B submissions used by every 2012 team.

### Next
`04_continuous_time_models.ipynb` will apply the identical protocol to Latent ODE and Neural CDE.
